# revit-api-rag Pipeline

数据准备阶段 — 在 Colab 中运行

## 流程
1. 克隆项目 & 安装依赖
2. 上传原始数据（API HTML + SDK 代码）
3. 解析 API 文档 → SQLite
4. 解析 SDK 代码 → SQLite (V2 pipeline: 项目发现 → ReadMe分析 → 匹配过滤 → Golden代码生成)
5. Embedding → ChromaDB
6. 下载生成的 .db 文件

## 首次使用 — 按顺序运行所有 Cell

1. **环境准备** — 安装依赖、设置 API Key、配置路径
2. **解压数据** — CHM → api_html，ZIP → sdk_samples
3. **解析数据** — HTML → SQLite，.cs → SQLite
4. **Embedding** — SQLite → ChromaDB 向量库，打包到 Drive
5. **测试 RAG** — 检索 + LLM 生成

## 再次使用 — 只需运行标记为 🔄 的 Cell

1. 🔄 环境准备
2. 🔄 恢复数据（从 Drive 解压 tar.gz）
3. 🔄 测试 RAG

## Step 0: 环境准备

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
from google.colab import drive
drive.mount('/content/drive')
# 克隆项目
#!git clone https://github.com/imkcrevit/revit-api-rag.git
%cd /content/drive/MyDrive/Colab_Projects/revit-api-rag/

!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab_Projects/revit-api-rag
check_api.py  data  legacy   pipeline	requirements-pipeline.txt  scripts
config	      docs  LICENSE  README.md	requirements-server.txt    server


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 安装 pipeline 依赖
!pip install -r requirements-pipeline.txt -q

In [10]:
# 验证关键包是否安装成功
import chromadb
import google.genai
import yaml
print("所有关键依赖已就绪 ✅")

所有关键依赖已就绪 ✅


添加OPENROUTER_API_KEY

In [ ]:
%%bash
cat << 'EOF' > .env
OPENROUTER_API_KEY=key

In [ ]:
import os
from dotenv import load_dotenv

# 从项目根目录加载 .env 文件
load_dotenv(dotenv_path=".env")

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    raise RuntimeError(
        "没有找到 OPENROUTER_API_KEY：\n"
        "- 请在项目根目录的 .env 文件中设置 OPENROUTER_API_KEY=你的 Key；\n"
        "- 或在系统环境变量中设置同名变量。"
    )

os.environ["OPENROUTER_API_KEY"] = api_key
print("API Key 已从 .env 设置 ✅")

API Key 已从 .env 设置 ✅


#### 转移旧文件 (已放弃)

In [ ]:
# 在 Colab 中运行，看看现在的情况
%cd /content/drive/MyDrive/Colab_Projects/revit-api-rag

# 1. 把旧文件移到 legacy/ 目录保留（不删除，以后迁移代码时参考）
!mkdir -p legacy
!mv main.ipynb legacy/
!mv split_revit.ipynb legacy/
!mv selct_all_api.ipynb legacy/
!mv setup.py legacy/
!mv requirements.txt legacy/
!mv union_merge_config.yaml legacy/
!mv output_text_1023.md legacy/
!mv project_dataset.json legacy/
!mv full_api.txt legacy/
!mv deepseek_tokenizer_v3 legacy/
!mv extra_data legacy/
!mv revit_sdk_prund legacy/
!mv revit_sdk_collection legacy/

# 2. 旧数据库移到 data/ 目录（后续还要用）
!mkdir -p data/legacy_db
!mv chromadb0815_api_1.db data/legacy_db/
!mv chromadb1022_code_1.db data/legacy_db/
!mv revit_api.db data/legacy_db/

# 3. 旧图片移到 docs/
!mkdir -p docs/images
!mv *.png docs/images/
!mv *.jpg docs/images/

# 4. 把新骨架从嵌套目录提升到根目录
!cp -r revit-api-rag/pipeline ./
!cp -r revit-api-rag/server ./
!cp -r revit-api-rag/config ./
!cp -r revit-api-rag/scripts ./
!cp revit-api-rag/requirements-pipeline.txt ./
!cp revit-api-rag/requirements-server.txt ./
!cp revit-api-rag/README.md ./README.md

# 5. 删除嵌套目录和压缩包
!rm -rf revit-api-rag/
!rm -f revit-api-rag-skeleton.tar.gz

# 6. 创建数据目录
!mkdir -p data/{chromadb,sqlite,knowledge,raw}

# 7. 验证最终结构
!echo "=== 项目根目录 ==="
!ls
!echo ""
!echo "=== pipeline/ ==="
!ls pipeline/
!echo ""
!echo "=== server/ ==="
!ls server/
!echo ""
!echo "=== legacy/ ==="
!ls legacy/


In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/revit_data/

!ls

In [ ]:
# 复制配置文件
!cp config/config.example.yaml config/config.yaml
print('配置文件已创建 ✅')
print('如需修改 embedding provider，请编辑 config/config.yaml')

## Step 1: 上传原始数据

### Git  Setup (每次打开同步git) 

In [48]:
import sys, importlib, random, re, os, subprocess
from pathlib import Path

# ── Parameters ───────────────────────────────────
HTML_DIR    = '/content/api_html/'
SAMPLE_N    = 500
QA_SAMPLE_N = 500
RANDOM_SEED = 42
# ─────────────────────────────────────────────────

PROJECT_ROOT = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'

# ── Step 0: Clean up & switch to main branch ─────
print('=== git setup ===')

# 0a. Delete all git hooks - Google Drive strips execute permission,
#     causing "cannot exec .git/hooks/post-checkout: Permission denied"
hooks_dir = Path(PROJECT_ROOT) / '.git/hooks'
for hook in hooks_dir.glob('*'):
    try:
        hook.unlink()
    except Exception:
        pass

# 0b. Remove stale index.lock (left behind by interrupted git operations)
lock_file = Path(PROJECT_ROOT) / '.git/index.lock'
if lock_file.exists():
    lock_file.unlink()
    print('Removed stale index.lock')

# 0c. Fetch latest main from origin
r_fetch = subprocess.run(
    ['git', '-C', PROJECT_ROOT, 'fetch', 'origin', 'main'],
    capture_output=True, text=True
)
print('fetch:', r_fetch.stdout.strip() or r_fetch.stderr.strip() or 'ok')

# 0d. Force-reset local main branch to origin/main
r_reset = subprocess.run(
    ['git', '-C', PROJECT_ROOT, 'checkout', '-B', 'main', 'origin/main'],
    capture_output=True, text=True
)
print('checkout:', (r_reset.stdout + r_reset.stderr).strip() or '(ok)')
# NOTE: Do NOT check r_reset.returncode here.
# "Reset branch main" is printed to stderr but is a SUCCESS message.
# The hook failure causes non-zero returncode even when checkout succeeded.

# Verify success by checking key files exist (reliable indicator)
qa_path  = Path(PROJECT_ROOT) / 'pipeline/api_parser/quality_agent.py'
llm_path = Path(PROJECT_ROOT) / 'pipeline/llm_client.py'
if not qa_path.exists():
    raise FileNotFoundError(
        f'quality_agent.py not found: {qa_path}\n'
        'git checkout may have failed. Try manually:\n'
        '  !rm -rf /content/drive/MyDrive/Colab_Projects/revit-api-rag/.git/hooks\n'
        '  !git -C /content/drive/MyDrive/Colab_Projects/revit-api-rag checkout -B main origin/main'
    )

# Show current HEAD
r_log = subprocess.run(
    ['git', '-C', PROJECT_ROOT, 'log', '--oneline', '-3'],
    capture_output=True, text=True
)
print('=== git log ===')
print(r_log.stdout)
print(f'quality_agent.py : {qa_path.exists()}')
print(f'llm_client.py    : {llm_path.exists()}')

=== git setup ===
fetch: From https://github.com/imkcrevit/revit-api-rag
 * branch            main       -> FETCH_HEAD
checkout: error: Your local changes to the following files would be overwritten by checkout:
	pipeline/run_all.ipynb
	pipeline/sdk_parser/extract.py
Please commit your changes or stash them before you switch branches.
Aborting
=== git log ===
57bb062 feat: add Step 6 notebook cell for DB snapshot save + Git LFS push from Colab
4688686 feat: enhance quality agent with progress bars for better tracking during execution
7f3896d add muti-thread prund

quality_agent.py : True
llm_client.py    : True


### 1.2如果没有开始训练使用这个Cell上传原始数据

In [ ]:
# 方式一：从 Google Drive 挂载
from google.colab import drive
drive.mount('/content/drive')

# 方式二：直接上传文件
# from google.colab import files
# uploaded = files.upload()

###  1.3 检查是否有解压文件，没有解压则开始解压


Cell2 - Colab 重新连接，找到打包的压缩文件\

In [ ]:
# 快速恢复数据（解压 tar.gz 到本地，几秒钟）
!tar -xzf /content/drive/MyDrive/Colab_Projects/revit_data/api_html.tar.gz -C /content/
!tar -xzf /content/drive/MyDrive/Colab_Projects/revit_data/sdk_samples.tar.gz -C /content/
print('数据恢复完成 ✅')

## Step 2: 解析 API 文档

#### 2.1 解压/压缩api html文件

##### 打包API

In [ ]:
# 3. 打包成一个 tar.gz（一个大文件写入 Drive 不会超时）
!tar -czf /content/drive/MyDrive/Colab_Projects/revit_data/api_html.tar.gz -C /content api_html/
print('API 打包完成 ✅')

##### 解压文件到目标位置

Cell-1 解压chm

In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/revit-api-rag

# 创建目录
!mkdir -p data/raw/api_html
!mkdir -p data/raw/sdk_samples

# 解压 CHM 到 Colab 本地（不是 Drive，速度快很多）
!apt-get install -y p7zip-full -q
!7z x "/content/drive/MyDrive/Colab_Projects/revit_data/RevitAPI.chm" -o/tmp/chm_out -y

# 看看解出来什么
!ls /tmp/chm_out/

#### 暂时屏蔽

In [31]:
# ── Step 1: Refresh import cache, clear stale modules ────
importlib.invalidate_caches()
for mod_name in list(sys.modules.keys()):
    if 'pipeline' in mod_name:
        del sys.modules[mod_name]

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.api_parser.parse_chm import parse_single_html
from pipeline.api_parser.quality_agent import run_quality_agent
import yaml

with open(f'{PROJECT_ROOT}/config/config.yaml') as f:
    config = yaml.safe_load(f)
print('\nModules loaded OK')

# ── ctor check (inline, no dependency on private functions) ──
def _is_ctor(title: str, fid: str) -> bool:
    fid_low = fid.lower()
    if '.#ctor' in fid_low or '#ctor(' in fid_low or fid_low.endswith('#ctor'):
        return True
    if re.search(r'\bconstructors?\s*$', title, re.IGNORECASE):
        return True
    return False

# ════════════════════════════════════════════════
# Phase 1: Random sample & parse
# ════════════════════════════════════════════════
all_files = list(Path(HTML_DIR).rglob('*.html')) + list(Path(HTML_DIR).rglob('*.htm'))
print(f'Total HTML files found: {len(all_files)}')

random.seed(RANDOM_SEED)
sample_files = random.sample(all_files, min(SAMPLE_N, len(all_files)))

results = []
skipped_ctor = 0
skipped_noise = 0

for f in sample_files:
    data = parse_single_html(str(f))
    if data:
        results.append(data)
    else:
        try:
            from bs4 import BeautifulSoup
            html = Path(f).read_text(encoding='utf-8', errors='ignore')
            soup = BeautifulSoup(html, 'html.parser')
            t = soup.find('title')
            title = t.string.strip() if t and t.string else ''
            m = soup.find('meta', attrs={'name': 'Microsoft.Help.F1'})
            fid = m.get('content', '') if m else ''
            if _is_ctor(title, fid):
                skipped_ctor += 1
            else:
                skipped_noise += 1
        except Exception:
            skipped_noise += 1

print(f'\nSampled {len(sample_files)} files -> valid: {len(results)}')
print(f'  Constructor (ctor) filtered : {skipped_ctor}')
print(f'  Other noise filtered        : {skipped_noise}\n')

# ── Field fill-rate stats ─────────────────────────────────
fields = ['name', 'full_id', 'namespace', 'summary', 'info', 'parameters', 'syntax', 'members', 'remark']
print('=== Field fill rate ===')
for field in fields:
    filled = sum(1 for r in results if r.get(field))
    pct = filled / len(results) * 100 if results else 0
    bar = chr(9608) * int(pct / 5) + chr(9617) * (20 - int(pct / 5))
    print(f'  {field:<14} {bar}  {filled:>4}/{len(results)}  ({pct:.1f}%)')

# ── First 10 records ──────────────────────────────────────
print('\n' + '='*70)
print('=== First 10 records ===')
print('='*70)

for i, item in enumerate(results[:10], 1):
    name     = item.get('name', '')
    full_id  = item.get('full_id', '')
    ns       = item.get('namespace', '')
    summary  = (item.get('summary') or item.get('info') or '')[:120]
    params   = item.get('parameters') or ''
    syntax   = (item.get('syntax') or '')[:80]
    member_n = len((item.get('members') or '').splitlines())

    print(f'\n[{i:02d}] {name}')
    print(f'     full_id   : {full_id}')
    print(f'     namespace : {ns}')
    print(f'     summary   : {summary or "(empty)"}')
    if params:
        plines = params.strip().splitlines()
        print(f'     params({len(plines)}) :')
        for pl in plines[:3]:
            print(f'       {pl}')
        if len(plines) > 3:
            print(f'       ... ({len(plines)} total)')
    if syntax:
        print(f'     syntax    : {syntax}...')
    if member_n:
        print(f'     members   : {member_n} members')
    print('-'*70)

# ── Random 5 deep checks ──────────────────────────────────
print('\n=== Random 5 deep checks ===')
for item in random.sample(results, min(5, len(results))):
    print(f'\n  [{item["name"]}]')
    print(f'  full_id : {item.get("full_id")}')
    print(f'  syntax  : {(item.get("syntax") or "")[:100]}')
    raw_p = item.get("parameters") or ""
    print(f'  params  : {raw_p[:200] if raw_p else "-"}')
    print(f'  summary : {(item.get("summary") or item.get("info") or "")[:150]}')

# ════════════════════════════════════════════════
# Phase 2: Quality Agent sample audit (Gemini -> Claude)
# ════════════════════════════════════════════════
print('\n\n' + '='*70)
print('  Quality Agent  (Gemini audit + Claude rewrite)')
print('='*70 + '\n')

random.seed(RANDOM_SEED + 1)
qa_sample = random.sample(results, min(QA_SAMPLE_N, len(results)))

qa_results = run_quality_agent(
    qa_sample, config,
    html_dir=HTML_DIR,
    max_stage2=200,
    verbose=True,
)

# ── QA result display ────────────────────────────────────
low_q    = [r for r in qa_results if r['_quality_score'] < 0.6]
rewrites = [r for r in qa_results if r['_rewritten']]
audit_failed = [r for r in qa_results if any('audit_failed' in iss for iss in r.get('_quality_issues', []))]

# --- Score distribution ---
print('\n' + '='*70)
print('=== Score Distribution ===')
print('='*70)
buckets = {'0.0-0.4': 0, '0.4-0.6': 0, '0.6-0.7': 0, '0.7-0.8': 0, '0.8-0.9': 0, '0.9-1.0': 0, '1.0': 0}
for r in qa_results:
    s = r['_quality_score']
    if s < 0.4:   buckets['0.0-0.4'] += 1
    elif s < 0.6: buckets['0.4-0.6'] += 1
    elif s < 0.7: buckets['0.6-0.7'] += 1
    elif s < 0.8: buckets['0.7-0.8'] += 1
    elif s < 0.9: buckets['0.8-0.9'] += 1
    elif s < 1.0: buckets['0.9-1.0'] += 1
    else:         buckets['1.0'] += 1

for label, cnt in buckets.items():
    bar = chr(9608) * cnt
    print(f'  {label:<9} {bar:<55} {cnt:>3}')

print(f'\n  HTML found    : {sum(1 for r in qa_results if r.get("_html_found"))} / {len(qa_results)}')
print(f'  audit_failed  : {len(audit_failed)}')
print(f'  needs Stage-2 : {len(low_q)} (threshold=0.6)')

# --- Sample: first 5 records with their scores & issues ---
print('\n' + '='*70)
print('=== Sample scores (first 5 records) ===')
print('='*70)
for r in qa_results[:5]:
    print(f"  [{r['_quality_score']:.2f}] {r.get('name','')}")
    print(f"        html={r.get('_html_found')}  issues={r.get('_quality_issues',[])}")
    print(f"        summary={repr((r.get('summary') or r.get('info') or '')[:80])}")
    print()

# --- Low quality detail ---
print('\n' + '='*70)
print(f'=== Low quality (score < 0.6): {len(low_q)} / {QA_SAMPLE_N} ===')
print('='*70)
for item in low_q[:8]:
    print(f"\n  [{item['name']}]  score={item['_quality_score']:.2f}  rewritten={item['_rewritten']}")
    print(f"  issues  : {'; '.join(item['_quality_issues'][:3])}")
    print(f"  summary : {(item.get('summary') or item.get('info') or '')[:100]}")
    if item.get('parameters'):
        print(f"  params  : {item['parameters'][:120]}")
    print('-'*60)

print(f'\nStage-2 (Claude) rewrites: {len(rewrites)}')


Modules loaded OK
Total HTML files found: 28863

Sampled 500 files -> valid: 480
  Constructor (ctor) filtered : 16
  Other noise filtered        : 4

=== Field fill rate ===
  name           ████████████████████   480/480  (100.0%)
  full_id        ████████████████████   480/480  (100.0%)
  namespace      ████████████████████   480/480  (100.0%)
  summary        ████████████████░░░░   395/480  (82.3%)
  info           ████░░░░░░░░░░░░░░░░   114/480  (23.8%)
  parameters     ██░░░░░░░░░░░░░░░░░░    69/480  (14.4%)
  syntax         ████████████████░░░░   394/480  (82.1%)
  members        █████░░░░░░░░░░░░░░░   126/480  (26.2%)
  remark         █████░░░░░░░░░░░░░░░   120/480  (25.0%)

=== First 10 records ===

[01] WorksharingDisplaySettings.GetAllUsersWithGraphicOverrides Method
     full_id   : WorksharingDisplaySettings.GetAllUsersWithGraphicOverrides
     namespace : Autodesk.Revit.DB
     summary   : Returns all usernames that have graphic overrides. This list consists of all users

KeyboardInterrupt: 

#### 2.2 复制API HTML到项目目录

In [ ]:
%cd /content

# 1. 解压到 Colab 本地磁盘（不经过 Drive，非常快）
!mkdir -p /content/api_html
!7z x "/content/drive/MyDrive/Colab_Projects/revit_data/RevitAPI.chm" -o/content/chm_tmp -y
!mv /content/chm_tmp/html/* /content/api_html/
!rm -rf /content/chm_tmp

# 2. 验证
!find /content/api_html/ -name "*.htm*" | wc -l
print('API 解压到本地完成 ✅')

#### 2.3测试api的检测率

In [ ]:
# ── Step 1: Refresh import cache, clear stale modules ────
importlib.invalidate_caches()
for mod_name in list(sys.modules.keys()):
    if 'pipeline' in mod_name:
        del sys.modules[mod_name]

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.api_parser.parse_chm import parse_single_html
from pipeline.api_parser.quality_agent import run_quality_agent
import yaml

with open(f'{PROJECT_ROOT}/config/config.yaml') as f:
    config = yaml.safe_load(f)
print('\nModules loaded OK')

# ── ctor check (inline, no dependency on private functions) ──
def _is_ctor(title: str, fid: str) -> bool:
    fid_low = fid.lower()
    if '.#ctor' in fid_low or '#ctor(' in fid_low or fid_low.endswith('#ctor'):
        return True
    if re.search(r'\bconstructors?\s*$', title, re.IGNORECASE):
        return True
    return False

# ════════════════════════════════════════════════
# Phase 1: Random sample & parse
# ════════════════════════════════════════════════
all_files = list(Path(HTML_DIR).rglob('*.html')) + list(Path(HTML_DIR).rglob('*.htm'))
print(f'Total HTML files found: {len(all_files)}')

random.seed(RANDOM_SEED)
sample_files = random.sample(all_files, min(SAMPLE_N, len(all_files)))

results = []
skipped_ctor = 0
skipped_noise = 0

for f in sample_files:
    data = parse_single_html(str(f))
    if data:
        results.append(data)
    else:
        try:
            from bs4 import BeautifulSoup
            html = Path(f).read_text(encoding='utf-8', errors='ignore')
            soup = BeautifulSoup(html, 'html.parser')
            t = soup.find('title')
            title = t.string.strip() if t and t.string else ''
            m = soup.find('meta', attrs={'name': 'Microsoft.Help.F1'})
            fid = m.get('content', '') if m else ''
            if _is_ctor(title, fid):
                skipped_ctor += 1
            else:
                skipped_noise += 1
        except Exception:
            skipped_noise += 1

print(f'\nSampled {len(sample_files)} files -> valid: {len(results)}')
print(f'  Constructor (ctor) filtered : {skipped_ctor}')
print(f'  Other noise filtered        : {skipped_noise}\n')

# ── Field fill-rate stats ─────────────────────────────────
fields = ['name', 'full_id', 'namespace', 'summary', 'info', 'parameters', 'syntax', 'members', 'remark']
print('=== Field fill rate ===')
for field in fields:
    filled = sum(1 for r in results if r.get(field))
    pct = filled / len(results) * 100 if results else 0
    bar = chr(9608) * int(pct / 5) + chr(9617) * (20 - int(pct / 5))
    print(f'  {field:<14} {bar}  {filled:>4}/{len(results)}  ({pct:.1f}%)')

# ── First 10 records ──────────────────────────────────────
print('\n' + '='*70)
print('=== First 10 records ===')
print('='*70)

for i, item in enumerate(results[:10], 1):
    name     = item.get('name', '')
    full_id  = item.get('full_id', '')
    ns       = item.get('namespace', '')
    summary  = (item.get('summary') or item.get('info') or '')[:120]
    params   = item.get('parameters') or ''
    syntax   = (item.get('syntax') or '')[:80]
    member_n = len((item.get('members') or '').splitlines())

    print(f'\n[{i:02d}] {name}')
    print(f'     full_id   : {full_id}')
    print(f'     namespace : {ns}')
    print(f'     summary   : {summary or "(empty)"}')
    if params:
        plines = params.strip().splitlines()
        print(f'     params({len(plines)}) :')
        for pl in plines[:3]:
            print(f'       {pl}')
        if len(plines) > 3:
            print(f'       ... ({len(plines)} total)')
    if syntax:
        print(f'     syntax    : {syntax}...')
    if member_n:
        print(f'     members   : {member_n} members')
    print('-'*70)

# ── Random 5 deep checks ──────────────────────────────────
print('\n=== Random 5 deep checks ===')
for item in random.sample(results, min(5, len(results))):
    print(f'\n  [{item["name"]}]')
    print(f'  full_id : {item.get("full_id")}')
    print(f'  syntax  : {(item.get("syntax") or "")[:100]}')
    raw_p = item.get("parameters") or ""
    print(f'  params  : {raw_p[:200] if raw_p else "-"}')
    print(f'  summary : {(item.get("summary") or item.get("info") or "")[:150]}')

# ════════════════════════════════════════════════
# Phase 2: Quality Agent sample audit (Gemini -> Claude)
# ════════════════════════════════════════════════
print('\n\n' + '='*70)
print('  Quality Agent  (Gemini audit + Claude rewrite)')
print('='*70 + '\n')

random.seed(RANDOM_SEED + 1)
qa_sample = random.sample(results, min(QA_SAMPLE_N, len(results)))

qa_results = run_quality_agent(
    qa_sample, config,
    html_dir=HTML_DIR,
    max_stage2=200,
    verbose=True,
)

# ── QA result display ────────────────────────────────────
low_q    = [r for r in qa_results if r['_quality_score'] < 0.6]
rewrites = [r for r in qa_results if r['_rewritten']]
audit_failed = [r for r in qa_results if any('audit_failed' in iss for iss in r.get('_quality_issues', []))]

# --- Score distribution ---
print('\n' + '='*70)
print('=== Score Distribution ===')
print('='*70)
buckets = {'0.0-0.4': 0, '0.4-0.6': 0, '0.6-0.7': 0, '0.7-0.8': 0, '0.8-0.9': 0, '0.9-1.0': 0, '1.0': 0}
for r in qa_results:
    s = r['_quality_score']
    if s < 0.4:   buckets['0.0-0.4'] += 1
    elif s < 0.6: buckets['0.4-0.6'] += 1
    elif s < 0.7: buckets['0.6-0.7'] += 1
    elif s < 0.8: buckets['0.7-0.8'] += 1
    elif s < 0.9: buckets['0.8-0.9'] += 1
    elif s < 1.0: buckets['0.9-1.0'] += 1
    else:         buckets['1.0'] += 1

for label, cnt in buckets.items():
    bar = chr(9608) * cnt
    print(f'  {label:<9} {bar:<55} {cnt:>3}')

print(f'\n  HTML found    : {sum(1 for r in qa_results if r.get("_html_found"))} / {len(qa_results)}')
print(f'  audit_failed  : {len(audit_failed)}')
print(f'  needs Stage-2 : {len(low_q)} (threshold=0.6)')

# --- Sample: first 5 records with their scores & issues ---
print('\n' + '='*70)
print('=== Sample scores (first 5 records) ===')
print('='*70)
for r in qa_results[:5]:
    print(f"  [{r['_quality_score']:.2f}] {r.get('name','')}")
    print(f"        html={r.get('_html_found')}  issues={r.get('_quality_issues',[])}")
    print(f"        summary={repr((r.get('summary') or r.get('info') or '')[:80])}")
    print()

# --- Low quality detail ---
print('\n' + '='*70)
print(f'=== Low quality (score < 0.6): {len(low_q)} / {QA_SAMPLE_N} ===')
print('='*70)
for item in low_q[:8]:
    print(f"\n  [{item['name']}]  score={item['_quality_score']:.2f}  rewritten={item['_rewritten']}")
    print(f"  issues  : {'; '.join(item['_quality_issues'][:3])}")
    print(f"  summary : {(item.get('summary') or item.get('info') or '')[:100]}")
    if item.get('parameters'):
        print(f"  params  : {item['parameters'][:120]}")
    print('-'*60)

print(f'\nStage-2 (Claude) rewrites: {len(rewrites)}')


#### 2.4: Quality Agent — Full Run (all API data → SQLite) 剪枝所有的API数据

In [ ]:
"""
Step 2.1 - Quality Agent PRODUCTION RUN
  - Runs on ALL api_data (parsed in Step 2 / Cell above)
  - Stage-1 Gemini : scores every record vs raw HTML  [tqdm bar]
  - Stage-2 Claude : rewrites low-quality records     [tqdm bar]
  - save_to_sqlite : batch INSERT with progress bar   [tqdm bar]
  - 403 failures   : source files logged to LOG_403
"""
import sys, importlib, os
from pathlib import Path

PROJECT_ROOT = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'
HTML_DIR     = '/content/api_html/'
DB_PATH      = f'{PROJECT_ROOT}/data/sqlite/revit_api.db'
LOG_403      = f'{PROJECT_ROOT}/data/sqlite/quality_agent_403.log'
MAX_STAGE2   = 5000   # max Claude rewrites per run

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

importlib.invalidate_caches()
for mod_name in list(sys.modules.keys()):
    if 'pipeline' in mod_name:
        del sys.modules[mod_name]

from pipeline.api_parser.quality_agent import run_quality_agent
from pipeline.api_parser.parse_chm import save_to_sqlite
import yaml
from tqdm.auto import tqdm

with open(f'{PROJECT_ROOT}/config/config.yaml') as f:
    config = yaml.safe_load(f)

# ── [1/3] Ensure api_data is available ───────────────────────
if 'api_data' not in dir() or not api_data:
    print('[1/3] api_data not found — parsing HTML files...')
    from pipeline.api_parser.parse_chm import parse_all_api_html
    with tqdm(desc='Parsing HTML', unit='file', dynamic_ncols=True) as pbar:
        api_data = parse_all_api_html(HTML_DIR)
        pbar.update(len(api_data))
    print(f'      Parsed {len(api_data)} records\n')
else:
    print(f'[1/3] api_data already loaded: {len(api_data)} records')

print()
print(f'[2/3] Running Quality Agent on ALL {len(api_data)} records...')
print(f'      Stage-1: Gemini | Stage-2: Claude | Max rewrites: {MAX_STAGE2}')
print(f'      DB path : {DB_PATH}')
print(f'      403 log : {LOG_403}')
print()

# ── [2/3] Run Quality Agent (tqdm bars inside run_quality_agent) ──
import inspect
_qa_params = inspect.signature(run_quality_agent).parameters
_qa_kwargs = dict(
    html_dir=HTML_DIR,
    max_stage2=MAX_STAGE2,
    verbose=True,
)
if 'failed_403_log_path' in _qa_params:
    _qa_kwargs['failed_403_log_path'] = LOG_403
else:
    print('⚠ Old quality_agent.py — run Step 0 (git pull) to enable 403 logging')
api_data_clean = run_quality_agent(api_data, config, **_qa_kwargs)

# ── [3/3] Save to SQLite (tqdm bar inside save_to_sqlite) ────────
print()
print('[3/3] Saving to SQLite...')
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f'      Removed old DB')

save_to_sqlite(api_data_clean, DB_PATH)

# ── Final stats ───────────────────────────────────────────────────
total      = len(api_data_clean)
low_q      = sum(1 for r in api_data_clean if r.get('_quality_score', 1.0) < 0.6)
rewrites   = sum(1 for r in api_data_clean if r.get('_rewritten', False))
html_hit   = sum(1 for r in api_data_clean if r.get('_html_found', False))
cnt_403    = sum(1 for r in api_data_clean
                 if any('403' in str(i) for i in (r.get('_quality_issues') or [])))

print()
print('=' * 60)
print('  Quality Agent — Production Run Complete')
print('=' * 60)
print(f'  Total records saved  : {total}')
print(f'  HTML matched         : {html_hit}  ({html_hit/total*100:.1f}%)')
print(f'  Low quality (<0.6)   : {low_q}  ({low_q/total*100:.1f}%)')
print(f'  Claude rewrites      : {rewrites}')
print(f'  Remaining 403 errors : {cnt_403}'  + (' ← check LOG_403' if cnt_403 else ' ✓'))
print(f'  DB written           : {DB_PATH}')
if cnt_403:
    print(f'  403 log              : {LOG_403}')
print('=' * 60)


#### 2.5 随机抽查db文件

In [17]:
import sqlite3, random, textwrap

DB_PATH = '/content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db'

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM revit_api")
total = cur.fetchone()[0]
print(f"Total records: {total}\n")

cur.execute("SELECT * FROM revit_api ORDER BY RANDOM() LIMIT 10")
rows = cur.fetchall()
conn.close()

SEP = "─" * 70
for i, row in enumerate(rows, 1):
    print(f"\n[{i:02d}] {row['name']}")
    print(f"     full_id   : {row['full_id']}")
    print(f"     namespace : {row['namespace']}")
    summary = row['summary'] or row['info'] or ''
    print(f"     summary   : {textwrap.shorten(summary, 120) if summary else '(empty)'}")
    if row['syntax']:
        print(f"     syntax    : {row['syntax'][:100]}")
    if row['parameters']:
        plines = row['parameters'].strip().splitlines()
        print(f"     params({len(plines):>2}) : {plines[0][:80]}")
        for pl in plines[1:3]:
            print(f"               : {pl[:80]}")
        if len(plines) > 3:
            print(f"               : ... ({len(plines)} total)")
    score = row['quality_score']
    rewritten = row['rewritten']
    if score is not None:
        print(f"     quality   : {score:.2f}  rewritten={rewritten}")
    if row['quality_issues']:
        print(f"     issues    : {row['quality_issues'][:120]}")
    print(SEP)

Total records: 27596


[01] DatumPlane.HasBubbleInView Method
     full_id   : DatumPlane.HasBubbleInView
     namespace : Autodesk.Revit.DB
     summary   : Identifies if the DatumPlane has bubble or not.
     syntax    : public bool HasBubbleInView ( DatumEnds datumEnd , View view )
     params( 2) : [datumEnd : DatumEnds]  - The end of the datum plane.
               : [view : View]  - The view on which the DatumPlane shows.
     quality   : 0.80  rewritten=0
     issues    : important content in the HTML (return value, exceptions) is absent from parsed fields
──────────────────────────────────────────────────────────────────────

[02] BuiltInFailures.CutFailures.CannotCutInstanceOutWarn Property
     full_id   : BuiltInFailures.CutFailures.CannotCutInstanceOutWarn
     namespace : Autodesk.Revit.DB
     summary   : Can't cut instance of [Symbol] out of Wall.
     syntax    : public static FailureDefinitionId CannotCutInstanceOutWarn { get ; }
     quality   : 0.95  rewritten=0
────

#### 随机抽查

In [18]:
import sqlite3

DB_PATH = '/content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db'

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM revit_api")
total = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM revit_api WHERE quality_issues LIKE '%403%'")
n_403 = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM revit_api WHERE quality_issues LIKE '%audit_failed%'")
n_failed = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM revit_api WHERE rewritten = 1")
n_rewritten = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM revit_api WHERE quality_score < 0.6")
n_low = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM revit_api WHERE quality_score IS NULL")
n_null = cur.fetchone()[0]

conn.close()

print(f"Total records        : {total}")
print(f"403 Forbidden issues : {n_403}  ({n_403/total*100:.1f}%)")
print(f"audit_failed (any)   : {n_failed}  ({n_failed/total*100:.1f}%)")
print(f"Low quality (<0.6)   : {n_low}  ({n_low/total*100:.1f}%)")
print(f"Claude rewritten     : {n_rewritten}  ({n_rewritten/total*100:.1f}%)")
print(f"No quality score     : {n_null}  ({n_null/total*100:.1f}%)  ← skipped by QA agent")
print()
print(f"Gemini success rate  : {(total - n_failed)/total*100:.1f}%  ({total - n_failed}/{total})")

Total records        : 27596
403 Forbidden issues : 0  (0.0%)
audit_failed (any)   : 1  (0.0%)
Low quality (<0.6)   : 2709  (9.8%)
Claude rewritten     : 2708  (9.8%)
No quality score     : 0  (0.0%)  ← skipped by QA agent

Gemini success rate  : 100.0%  (27595/27596)


#### 2.6 清除数据库，并重新保存

In [5]:
!rm -f /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit-api-0310//revit_api.db

#save_to_sqlite(api_data, '/content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db')
#print(f'\nAPI 解析完成：{len(api_data)} 条 ✅')

## Step 3: 解析 SDK 代码

#### 3.1 打包SDK压缩包

In [ ]:
# 5. SDK 打包
!tar -czf /content/drive/MyDrive/Colab_Projects/revit_data/sdk_samples.tar.gz -C /content sdk_samples/
print('SDK 打包完成 ✅')

#### 3.2解压sdk到本地

In [20]:
# 4. SDK 解压到本地
!mkdir -p /content/sdk_samples
!unzip -o -q "/content/drive/MyDrive/Colab_Projects/revit_data/Samples.zip" -d /content/sdk_samples/

# 验证
!find /content/sdk_samples/ -name "*.cs" | wc -l
print('SDK 解压到本地完成 ✅')

1668
SDK 解压到本地完成 ✅


#### 3.3 验证SDK路径

In [49]:
# 验证
!find /content/sdk_samples/ -name "*.cs" | wc -l
print('SDK 解压完成 ✅')

1668
SDK 解压完成 ✅


#### 3.4 测试抽取样本进行检测成果

In [ ]:
"""
SDK 5-project sample inspector
Paste into a new Colab Code Cell and run directly.
"""
import sys, importlib, random
from pathlib import Path

PROJECT_ROOT = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'
SDK_ROOT     = '/content/sdk_samples/'   # adjust if your path differs

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

importlib.invalidate_caches()
for m in list(sys.modules.keys()):
    if 'pipeline' in m:
        del sys.modules[m]

import shutil, yaml

# ── Clear __pycache__ so Python reloads .py files, not stale .pyc ────
for _pc in Path(PROJECT_ROOT).rglob('__pycache__'):
    shutil.rmtree(_pc, ignore_errors=True)

import pipeline.sdk_parser.extract as _extract_mod
from pipeline.sdk_parser.extract import extract_all_sdk_projects, extract_cs_files

# ── Monkey-patch parse_code_blocks with the AST-walk version ─────────
# This works regardless of which .py/.pyc version was loaded from disk,
# because extract_cs_files looks up parse_code_blocks in the module namespace
# at call time (not at import time).
def _parse_code_blocks_fixed(code: str):
    try:
        from tree_sitter import Language, Parser
        import tree_sitter_c_sharp
    except Exception:
        return _extract_mod._fallback_prune_code(code)
    cleaned = code.lstrip('\ufeff')
    try:
        lang   = Language(tree_sitter_c_sharp.language())
        parser = Parser(lang)
        tree   = parser.parse(bytes(cleaned, 'utf8'))
    except Exception:
        return _extract_mod._fallback_prune_code(code)
    blocks, stack = [], [tree.root_node]
    while stack:
        node = stack.pop()
        if node.type == 'method_declaration':
            t = cleaned[node.start_byte:node.end_byte].strip()
            if t:
                blocks.append(t)
            continue
        stack.extend(reversed(node.children))
    return blocks if blocks else _extract_mod._fallback_prune_code(code)

_extract_mod.parse_code_blocks = _parse_code_blocks_fixed
print('parse_code_blocks: patched (AST walk, tree-sitter version-safe)')

with open(f'{PROJECT_ROOT}/config/config.yaml') as f:
    config = yaml.safe_load(f)

# ── 1. Discover projects ──────────────────────────────────────
sdk_root = Path(SDK_ROOT)
all_projects = [d for d in sdk_root.iterdir() if d.is_dir()]
print(f'Total SDK projects found: {len(all_projects)}')

random.seed(42)
sample_projects = random.sample(all_projects, min(5, len(all_projects)))

SEP = '─' * 72

# ── 2. Extract & show pruning diff ───────────────────────────
all_items = []
for proj_dir in sample_projects:
    items = extract_cs_files(str(proj_dir))
    all_items.extend(items)

    cs_files = list(proj_dir.rglob('*.cs'))
    readme_exists = any((proj_dir / n).exists() for n in ['ReadMe.rtf','ReadMe.txt','README.md'])
    print(f'\n[PROJECT] {proj_dir.name}')
    print(f'  .cs files  : {len(cs_files)}')
    print(f'  readme     : {"yes" if readme_exists else "no"}')

    for item in items:
        orig_lines  = item['code'].count('\n') + 1
        clean_lines = item['clean_code'].count('\n') + 1 if item['clean_code'] else 0
        ratio = clean_lines / orig_lines * 100 if orig_lines else 0
        print(f'  {item["filename"]:<45} {orig_lines:>4} → {clean_lines:>4} lines  ({ratio:.0f}%)')

        # Show first 10 lines of clean_code
        if item['clean_code']:
            preview = '\n'.join(item['clean_code'].splitlines()[:10])
            print('    ── clean_code preview ──')
            for ln in preview.splitlines():
                print(f'    {ln}')
            if clean_lines > 10:
                print(f'    ... ({clean_lines - 10} more lines)')
        else:
            print('    ── clean_code: (empty after pruning)')
    print(SEP)

# ── 3. LLM summarization (Claude) ────────────────────────────
print(f'\n\nRunning LLM summarization on {len(all_items)} files...\n')
print('=' * 72)

from pipeline.llm_client import create_llm_client
client = create_llm_client(config)

for item in all_items:
    code    = item.get('clean_code') or item.get('code') or ''
    readme  = (item.get('readme') or '')[:800]
    project = item.get('project', '')
    fname   = item.get('filename', '')

    if not code.strip():
        print(f'[SKIP] {project}/{fname}  (empty after pruning)')
        continue

    # clean_code is already pruned to method bodies only — send it in full.
    # Fall back to the raw file if clean_code is empty, capped at 20 000 chars
    # to stay well within Claude's context window while covering the whole file.
    code_for_llm = clean_code if clean_code.strip() else code[:20_000]
    prompt = (
        "You are a Revit SDK code analyst. "
        "Write a concise 2-3 sentence English description of this C# sample: "
        "what scenario it demonstrates, which main Revit APIs are involved, and why it is useful.\n\n"
        + (f"Project: {project} / {fname}\n" if project else "")
        + (f"ReadMe:\n{readme}\n\n" if readme else "")
        + f"Code (pruned, full):\n{code_for_llm}\n\n"
        + "Output the description text only. No JSON, no code, no bullet points."
    )

    try:
        desc = client.generate_text(prompt).strip()
    except Exception as e:
        desc = f'(LLM error: {e})'

    orig_lines  = item['code'].count('\n') + 1
    clean_lines = item['clean_code'].count('\n') + 1 if item['clean_code'] else 0

    print(f'\n[{project} / {fname}]')
    print(f'  Pruning : {orig_lines} → {clean_lines} lines  ({clean_lines/orig_lines*100:.0f}% retained)')
    print(f'  Summary : {desc}')
    print(SEP)

print('\nDone.')

### Step 3a: SDK Pipeline V2 — 项目发现 → ReadMe分析 → 匹配过滤 → Golden代码生成

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Step 3b: SDK Pipeline V2
#   Phase 0 : 项目发现 (os.walk + CS目录 + 忽略 boilerplate)
#   Phase 1 : ReadMe 分析 (Gemini Flash, parallel, tqdm)
#   Phase 1b: 文件匹配 + tree-sitter 方法抽取
#   Phase 2 : Golden Code 生成 (Claude, parallel, tqdm)
#   Phase 3 : 保存到 SQLite sdk_info 表
# ══════════════════════════════════════════════════════════════════
import sys, importlib, shutil, yaml
from pathlib import Path

PROJECT_ROOT = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'
SDK_ROOT     = '/content/sdk_samples/Samples'
DB_PATH      = f'{PROJECT_ROOT}/data/sqlite/revit_sdk.db'

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Clear stale bytecode so fresh extract.py V2 is loaded
for _pc in Path(PROJECT_ROOT).rglob('__pycache__'):
    shutil.rmtree(_pc, ignore_errors=True)
importlib.invalidate_caches()
for mod_name in list(sys.modules.keys()):
    if 'pipeline' in mod_name:
        del sys.modules[mod_name]

with open(f'{PROJECT_ROOT}/config/config.yaml') as f:
    config = yaml.safe_load(f)

from pipeline.sdk_parser.extract import run_sdk_pipeline, save_sdk_to_sqlite

# ── Run the full V2 pipeline ──────────────────────────────────
sdk_results = run_sdk_pipeline(
    sdk_root=SDK_ROOT,
    config=config,
    num_workers_phase1=config.get('sdk', {}).get('num_workers_phase1', 10),
    num_workers_phase2=config.get('sdk', {}).get('num_workers_phase2', 5),
    verbose=True,
)

# ── Save to SQLite (sdk_info table) ──────────────────────────
save_sdk_to_sqlite(sdk_results, DB_PATH)

# ── Sample output ─────────────────────────────────────────────
print()
print('Sample golden snippets (first 3):')
print('-' * 60)
for item in sdk_results[:3]:
    print(f"  Project : {item.get('project_name')}")
    print(f"  Summary : {(item.get('summary') or '')[:120]}")
    print(f"  APIs    : {(item.get('mentioned_apis') or '')[:80]}")
    print('-' * 60)



SDK Pipeline — sdk_root: /content/sdk_samples/Samples
  Stage 1 model (ReadMe): google/gemini-3-flash-preview
  Stage 2 model (Code):   anthropic/claude-sonnet-4.6
Phase 0: found 181 C# project directories under /content/sdk_samples/Samples


Discovering projects:   0%|          | 0/181 [00:00<?, ?dir/s]

Phase 0 complete: 181 valid projects with .cs files

Phase 1: ReadMe analysis — 181 projects | 10 threads


ReadMe analysis:   0%|          | 0/181 [00:00<?, ?proj/s]


Phase 1b: Matching files and extracting methods
  [1/181] SpanDirection — 1 files, 4 classes
    [1/4] class: Command
      ✓ found in Command.cs (2 methods)
    [2/4] class: Execute
      ✓ found in Command.cs (1 methods)
    [3/4] class: SpanDirectionAngle
  [miss] class: SpanDirectionAngle in SpanDirection
    [4/4] class: SpanDirectionSymbols
  [miss] class: SpanDirectionSymbols in SpanDirection
  [2/181] FabricationPartLayout — 17 files, 8 classes
    [1/8] class: FabricationPartLayout
      ✓ found in FabricationPartLayout.cs (14 methods)
    [2/8] class: OptimizeStraights
      ✓ found in OptimizeStraights.cs (1 methods)
    [3/8] class: StretchAndFit
      ✓ found in StretchAndFit.cs (3 methods)
    [4/8] class: ConvertToFabrication
      ✓ found in ConvertToFabrication.cs (1 methods)
    [5/8] class: PartRenumber
      ✓ found in PartRenumber.cs (4 methods)
    [6/8] class: Execute
  [miss] class: Execute in FabricationPartLayout
    [7/8] class: Optimize Lengths
  [miss] cla

Golden code generation:   0%|          | 0/164 [00:00<?, ?proj/s]

  [Claude JSON error] FabricationPartLayout: Unterminated string starting at: line 3 column 14 (char 373)
  [Claude JSON error] FreeFormElement: Expecting ',' delimiter: line 3 column 354 (char 646)
  [Claude JSON error] ShaftHolePuncher: Expecting ',' delimiter: line 3 column 369 (char 611)
  [Claude JSON error] GeometryAPI.GeometryCreation_BooleanOperation: Expecting ',' delimiter: line 3 column 391 (char 720)
  [Claude JSON error] GeometryAPI.UpdateExternallyTaggedBRep: Unterminated string starting at: line 3 column 14 (char 297)
  [Claude JSON error] AvoidObstruction: Unterminated string starting at: line 3 column 14 (char 383)
  [Claude JSON error] DuplicateViews: Expecting ',' delimiter: line 3 column 481 (char 780)
  [Claude JSON error] NewOpenings: Expecting ',' delimiter: line 3 column 409 (char 638)
  [Claude JSON error] CurtainSystem: Expecting ',' delimiter: line 3 column 353 (char 625)
  [Claude JSON error] AnalysisVisualizationFramework.SpatialFieldGradient: Expecting ','

Saving to SQLite:   0%|          | 0/153 [00:00<?, ?rec/s]

Saved 153 records to /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_sdk.db

Sample golden snippets (first 3):
------------------------------------------------------------
  Project : SpanDirection
  Summary : Demonstrates how to retrieve and display a structural floor slab's span direction angle and associated span direction sy
  APIs    : ["Autodesk.Revit.UI.IExternalCommand", "Autodesk.Revit.DB.ElementSet", "Autodesk
------------------------------------------------------------
  Project : CloudAPISample
  Summary : Demonstrates how to register and run a BIM 360 model migration sample using the Revit Cloud API by initializing a Sample
  APIs    : ["IExternalApplication", "IExternalCommand"]
------------------------------------------------------------
  Project : ModelessDialog.ModelessForm_IdlingEvent
  Summary : Demonstrates the modeless dialog request-handler pattern in the Revit API, using Idling-safe thread communication via In
  APIs    : ["Autodesk.Revit.D

#### Step 3b-check: 验证类匹配正确性（与 3b 完全一致的算法）

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Step 3b-check: 验证类匹配正确性（与 3b 完全一致的算法）
#   - 调用 match_and_extract 确保逻辑一致
#   - 无 Claude 总结，仅验证 match
#   - 最终报告前 3 个失败案例
# ══════════════════════════════════════════════════════════════════
import sys, importlib, shutil, yaml, re, io
from pathlib import Path
from contextlib import redirect_stdout
from tqdm.auto import tqdm

PROJECT_ROOT = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'
SDK_ROOT     = '/content/sdk_samples/Samples'

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Clear stale bytecode
for _pc in Path(PROJECT_ROOT).rglob('__pycache__'):
    shutil.rmtree(_pc, ignore_errors=True)
importlib.invalidate_caches()
for mod_name in list(sys.modules.keys()):
    if 'pipeline' in mod_name:
        del sys.modules[mod_name]

with open(f'{PROJECT_ROOT}/config/config.yaml') as f:
    config = yaml.safe_load(f)

from pipeline.sdk_parser.extract import (
    discover_sdk_projects, analyze_readme, match_and_extract
)
from pipeline.llm_client import create_llm_client

sdk_cfg = config.get("sdk", {})
gemini_client = create_llm_client(
    config, provider_override=sdk_cfg.get("stage1_provider", "gemini_flash")
)
gemini_client.max_tokens = 1024

# ── Phase 0: 发现项目 ──
projects = discover_sdk_projects(SDK_ROOT)

# ── Phase 1 + 1b: ReadMe 分析 + match_and_extract 验证 ──
MAX_REPORT = 3
unmatched_report = []
stats = {"total_projects": len(projects), "total_classes": 0,
         "matched_projects": 0, "missed_projects": 0, "no_readme": 0,
         "matched_classes": 0, "missed_classes": 0}

for proj in tqdm(projects, desc="项目进度", unit="proj"):
    proj_name = proj["project_name"]

    readme_analysis = analyze_readme(proj, gemini_client)
    if readme_analysis is None:
        stats["no_readme"] += 1
        continue

    key_classes = readme_analysis.get("key_classes_and_methods", [])
    stats["total_classes"] += len(key_classes)

    # 捕获 match_and_extract 的 print 输出来解析 miss 信息
    buf = io.StringIO()
    with redirect_stdout(buf):
        matched = match_and_extract(proj, readme_analysis)
    output = buf.getvalue()

    # 解析输出，统计匹配情况
    for line in output.splitlines():
        line = line.strip()
        # 兼容两种格式: [miss] 和 [match]
        if line.startswith("[miss]") or line.startswith("[match]"):
            stats["missed_classes"] += 1
            tqdm.write(f"  ✗ {line}")

            if len(unmatched_report) < MAX_REPORT:
                miss_match = re.search(r'class:\s*(.+?)\s+in\s+(.+)', line)
                class_name = miss_match.group(1) if miss_match else "?"
                actual_classes = set()
                for fpath in proj["all_cs_files"]:
                    try:
                        code_text = fpath.read_text(encoding="utf-8-sig", errors="ignore")
                        for m in re.finditer(r'\bclass\s+(\w+)', code_text):
                            actual_classes.add(m.group(1))
                    except Exception:
                        pass
                unmatched_report.append({
                    "project": proj_name,
                    "missing_class": class_name,
                    "normalized": re.sub(r'\s+', '', class_name).lower(),
                    "actual_files": [p.name for p in proj["all_cs_files"]],
                    "actual_classes": sorted(actual_classes),
                })

    # 匹配成功的类 = 总类数 - 失败数（从本次输出统计）
    missed_this = sum(1 for l in output.splitlines()
                      if l.strip().startswith("[miss]") or l.strip().startswith("[match]"))
    matched_this = len(key_classes) - missed_this
    stats["matched_classes"] += matched_this

    if matched is not None:
        stats["matched_projects"] += 1
        tqdm.write(f"  ✓ {proj_name} — {len(matched['code_details'])} classes matched, {missed_this} missed")
    else:
        stats["missed_projects"] += 1
        if missed_this > 0:
            tqdm.write(f"  ✗ {proj_name} — all {len(key_classes)} classes failed")

# ── 汇总报告 ──
print(f"\n{'='*60}")
print(f"匹配验证结果（使用 match_and_extract 一致算法）")
print(f"{'='*60}")
print(f"  项目总数:         {stats['total_projects']}")
print(f"  无 ReadMe:        {stats['no_readme']}")
print(f"  项目匹配成功:     {stats['matched_projects']}")
print(f"  项目匹配失败:     {stats['missed_projects']}")
print(f"  ──────────────────")
print(f"  类总数:           {stats['total_classes']}")
print(f"  类匹配成功:       {stats['matched_classes']}")
print(f"  类匹配失败:       {stats['missed_classes']}")
rate = stats['matched_classes'] / max(stats['total_classes'], 1) * 100
print(f"  类匹配成功率:     {rate:.1f}%")

if unmatched_report:
    print(f"\n{'─'*60}")
    print(f"失败详情（前 {MAX_REPORT} 个）")
    print(f"{'─'*60}")
    for i, item in enumerate(unmatched_report, 1):
        print(f"\n[{i}] Project: {item['project']}")
        print(f"    LLM 返回类名:  {item['missing_class']}")
        print(f"    标准化后:       {item['normalized']}")
        print(f"    实际 .cs 文件:  {item['actual_files']}")
        print(f"    实际类名列表:   {item['actual_classes']}")
else:
    print("\n所有类均已匹配成功！")


#### 强制colab更新

In [ ]:
!git checkout -- .
!git pull

## Step 4: Embedding 向量化

设置APIKEY All From OpenRouter

In [ ]:
import os
from google.colab import userdata

os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
print('OpenRouter API Key 已设置 ✅')

In [ ]:
# 测试 embedding 是否正常
import importlib
import config as config_module
importlib.reload(config_module)
from config import load_config
from pipeline.embedder.providers import create_embedding

config = load_config('/content/drive/MyDrive/Colab_Projects/revit-api-rag/config/config.yaml')
embedder = create_embedding(config)

# 测试一条
test = embedder.embed_query("Create structural column in Revit")
print(f'Model: {embedder.model_name}')
print(f'Dimension: {len(test)}')
print(f'前5个值: {test[:5]}')
print('Embedding 测试成功 ✅')

Model: openai/text-embedding-3-large
Dimension: 3072
前5个值: [-0.005171133670955896, -0.0509759746491909, -0.018134385347366333, 0.011866958811879158, 0.028011150658130646]
Embedding 测试成功 ✅


开始Embedding

In [ ]:
from config import load_config
from pipeline.embedder.embed import embed_api_data, embed_code_data

config = load_config('/content/drive/MyDrive/Colab_Projects/revit-api-rag/config/config.yaml')
version = config.get('revit_version', '2026')

print(f'Embedding provider: {config["embedding"]["provider"]}')
print(f'Revit version: {version}')
print('开始向量化...')

Embedding provider: openai
Revit version: 2026
开始向量化...


In [ ]:
base = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'

# API 向量化
embed_api_data(
    config=config,
    api_db_path=f'{base}/data/sqlite/revit_api.db',
    chromadb_dir='/content/chromadb_api/',
)
print('API 向量化完成 ✅')

从 /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db 读取 27596 条 API 数据
  API embedding 进度: 50/27596
  API embedding 进度: 550/27596
  API embedding 进度: 1050/27596
  API embedding 进度: 1550/27596
  API embedding 进度: 2050/27596
  API embedding 进度: 2550/27596
  API embedding 进度: 3050/27596
  API embedding 进度: 3550/27596
  API embedding 进度: 4050/27596
  API embedding 进度: 4550/27596
  API embedding 进度: 5050/27596
  API embedding 进度: 5550/27596
  API embedding 进度: 6050/27596
  API embedding 进度: 6550/27596
  API embedding 进度: 7050/27596
  API embedding 进度: 7550/27596
  API embedding 进度: 8050/27596
  API embedding 进度: 8550/27596
  API embedding 进度: 9050/27596
  API embedding 进度: 9550/27596
  API embedding 进度: 10050/27596
  API embedding 进度: 10550/27596
  API embedding 进度: 11050/27596
  API embedding 进度: 11550/27596
  API embedding 进度: 12050/27596
  API embedding 进度: 12550/27596
  API embedding 进度: 13050/27596
  API embedding 进度: 13550/27596
  API embedding 进度: 14050/2759

In [ ]:
# SDK 向量化
import importlib
import pipeline.embedder.embed as embed_module
importlib.reload(embed_module)
from pipeline.embedder.embed import embed_code_data

embed_code_data(
    config=config,
    sdk_db_path=f'{base}/data/sqlite/revit_sdk.db',
    chromadb_dir='/content/chromadb_code/',
)
print('SDK 向量化完成 ✅')

从 /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_sdk.db 读取 153 条 SDK 数据 [sdk_info (summary-only)]
  Code embedding 进度: 20/153
已写入 /content/chromadb_code/meta.json
Code 向量化完成，共 153 条，存入 /content/chromadb_code/
SDK 向量化完成 ✅


In [ ]:
# 完成后打包到 Drive
!tar -czf {base}/data/chromadb_code.tar.gz -C /content chromadb_code/
print('SDK 向量库已保存到 Drive ✅')

SDK 向量库已保存到 Drive ✅


打包回Drive

In [ ]:
# 打包存回 Drive
!tar -czf {base}/data/chromadb_api.tar.gz -C /content chromadb_api/
!tar -czf {base}/data/chromadb_code.tar.gz -C /content chromadb_code/
print('向量库已保存到 Drive ✅')

## Step 5: 验证 & 下载

In [ ]:
import chromadb
import json

paths = {
    'api': '/content/chromadb_api/',
    'code': '/content/chromadb_code/',
}

for db_type, db_dir in paths.items():
    meta_path = f'{db_dir}/meta.json'
    try:
        with open(meta_path) as f:
            meta = json.load(f)
        print(f'\n{db_type.upper()} 向量库:')
        print(f'  Provider: {meta["embedding_provider"]}')
        print(f'  Model: {meta["embedding_model"]}')
        print(f'  Dimension: {meta["embedding_dimension"]}')
        print(f'  Records: {meta["record_count"]}')

        client = chromadb.PersistentClient(path=db_dir)
        for col in client.list_collections():
            print(f'  Collection: {col.name}, Count: {col.count()}')
    except FileNotFoundError:
        print(f'\n{db_type.upper()} 向量库: 未找到，需要重新生成')


API 向量库:
  Provider: openai
  Model: openai/text-embedding-3-large
  Dimension: 3072
  Records: 27596
  Collection: revit_api, Count: 27596

CODE 向量库:
  Provider: openai
  Model: openai/text-embedding-3-large
  Dimension: 3072
  Records: 153
  Collection: revit_sdk, Count: 153


In [ ]:
# 打包数据文件用于下载
!tar -czf revit_rag_data.tar.gz data/
print('数据已打包: revit_rag_data.tar.gz')
print(f'文件大小: {os.path.getsize("revit_rag_data.tar.gz") / 1024 / 1024:.1f} MB')

# 下载
from google.colab import files
files.download('revit_rag_data.tar.gz')

# 或者保存到 Google Drive
# !cp revit_rag_data.tar.gz /content/drive/MyDrive/

## 测试数据库db文件准确度

In [ ]:
# Cell 1: 加载两层检索器 (ChromaDB 语义索引 + SQLite 内容仓库)
from config import load_config
from pipeline.retriever import RAGRetriever

base = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'
config = load_config(f'{base}/config/config.yaml')

retriever = RAGRetriever(
    config=config,
    api_db_path=f'{base}/data/sqlite/revit_api.db',
    sdk_db_path=f'{base}/data/sqlite/revit_sdk.db',
    chromadb_api_dir='/content/chromadb_api/',
    chromadb_code_dir='/content/chromadb_code/',
)

print(f"API 向量库: {retriever._api_collection.count()} 条")
print(f"SDK 向量库: {retriever._code_collection.count()} 条")
print(f"SDK schema: {'sdk_info (V2)' if retriever._sdk_new_schema else 'revit_sdk (legacy)'}")
print('两层检索器加载完成 ✅')

API 向量库: 27596 条
SDK 向量库: 153 条
SDK schema: sdk_info (V2)
两层检索器加载完成 ✅


In [ ]:
# Cell 2: 检索 (两层: ChromaDB 搜索 → SQLite 回查全文)
# retriever.search() 内部:
#   1. embed query → ChromaDB 语义搜索 → 返回 IDs + distances
#   2. 用 IDs 批量查询 SQLite → 获取完整 content/info/syntax 等字段
#   3. 组装成 RetrievedItem 列表

def search_and_show(query, api_top_k=15, code_top_k=5):
    results = retriever.search(query, api_top_k=api_top_k, code_top_k=code_top_k)
    print(retriever.format_results(results))
    return results

In [ ]:
# Cell 3: 测试检索
query = '创建结构柱'
results = search_and_show(query)

API 检索结果：

[1] 相似度: -0.2401
    名称: CompoundStructure
    摘要: Describes the internal structure of a wall, floor, roof or ceiling....

[2] 相似度: -0.2504
    名称: BuiltInFailures.FamilyFailures.StructuralColumnAttachedToNonStructuralTarget
    摘要: The structural column is attached to a non-structural target....

[3] 相似度: -0.2643
    名称: ParameterTypeId.CeilingStructureIdParam
    摘要: "Structure"...

[4] 相似度: -0.2781
    名称: ParameterTypeId.StructuralAnalyticalColumnRigidLink
    摘要: "Analytical Links"...

[5] 相似度: -0.2783
    名称: CompoundStructure.CreateSimpleCompoundStructure
    摘要: Creates a non-vertically compound structure comprised of parallel layers....

[6] 相似度: -0.2937
    名称: GroupTypeId.Structural
    摘要: Structural....

[7] 相似度: -0.2957
    名称: ParameterTypeId.StructuralSectionCantileverHeight
    摘要: "Cantilever Height"...

[8] 相似度: -0.2996
    名称: ParameterTypeId.StructuralSectionCommonCentroidVertical
    摘要: "Centroid Vertical"...

[9] 相似度: -0.3023
    名称: ParameterTypeId.S

In [ ]:
# Cell 4: RAG 生成 (两层检索 + LLM)
from openai import OpenAI
import os

llm_client = OpenAI(
    api_key=os.environ['OPENROUTER_API_KEY'],
    base_url='https://openrouter.ai/api/v1',
)

def generate_code(query: str, results=None, model: str = 'google/gemini-2.5-flash'):
    """RAG 生成: 两层检索结果 + LLM 生成代码"""
    if results is None:
        results = retriever.search(query, api_top_k=15, code_top_k=5)

    # build_context 从 SQLite 获取完整 content/syntax/parameters
    ctx = retriever.build_context(results)

    system_prompt = '''You are a professional BIM engineer expert in Revit API.
You write C# code for Revit plugins following these standards:
1. Completeness: Include the entire process from start to finish
2. Professionalism: Correctly handle Revit element characteristics
3. Robustness: Include error handling and boundary condition checking
4. Scalability: Easy to extend
5. Best practice: Follow Revit API development specifications

Give the user a complete, working C# plugin code solution.'''

    user_prompt = f'''User question: {query}

Revit API Reference:
{ctx['api_context']}

SDK Code Reference:
{ctx['code_context']}

Based on the references above, generate a complete Revit C# plugin to solve the user's question.
Include all necessary using statements, the IExternalCommand class, and proper Transaction handling.'''

    response = llm_client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0.3,
        max_tokens=4096,
    )

    return response.choices[0].message.content


# 运行完整 RAG 流程
query = '创建结构柱'
print(f'查询: {query}\n')

results = retriever.search(query, api_top_k=15, code_top_k=5)
print(f'检索到 API: {len(results.api_items)} 条, SDK: {len(results.sdk_items)} 条')

print('\n生成中...\n')
answer = generate_code(query, results)
print(answer)

## 完成！

下一步：
1. 将 `revit_rag_data.tar.gz` 上传到 GCP 服务器
2. 解压到项目的 `data/` 目录
3. 运行 `python -m server.app.main` 启动服务